In [61]:
import os
import json
import pandas as pd
import traceback

In [62]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [63]:
from dotenv import load_dotenv

load_dotenv()

True

In [65]:
KEY=os.getenv("GOOGLE_API_KEY")

In [101]:
llm=ChatGoogleGenerativeAI(google_api_key=KEY,model="gemini-1.5-pro-latest", temperature=0.5)

In [83]:
llm

ChatGoogleGenerativeAI(profile={'max_input_tokens': 1000000, 'max_output_tokens': 8192, 'image_inputs': True, 'audio_inputs': True, 'video_inputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'image_url_inputs': True, 'pdf_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-1.5-flash', temperature=0.5, client=<google.genai.client.Client object at 0x000001C2217A5580>, default_metadata=(), model_kwargs={})

In [100]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence
from langchain_core.output_parsers import StrOutputParser

import PyPDF2





In [85]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here"
        },
        "correct": "correct answer key"
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here"
        },
        "correct": "correct answer key"
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here"
        },
        "correct": "correct answer key"
    }
}


In [86]:
template = """
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz of {number} multiple choice questions for {subject} students in {tone} tone. 
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [87]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=template
)

In [88]:
template2 = """
You are an expert English grammarian and writer. Given this quiz:
{quiz}
Evaluate the complexity for {subject} students. If it's not at par with their abilities, 
suggest improvements or updates. Keep your analysis under 50 words.
"""

In [89]:
quiz_evaluation_prompt = PromptTemplate(
    input_variables=["subject", "quiz"],
    template=template2)

In [90]:
# --- REPLACING LLMChain and SequentialChain with LCEL ---

# 1. The Generation Chain
# This replaces quiz_chain
quiz_chain = quiz_generation_prompt | llm | StrOutputParser()

# 2. The Evaluation Chain
# This replaces review_chain
review_chain = quiz_evaluation_prompt | llm | StrOutputParser()

In [91]:
def generate_mcqs(text, number, subject, tone):

        # Step 1: Run the generator
        quiz_output = quiz_chain.invoke({
            "text": text,
            "number": number,
            "subject": subject,
            "tone": tone,
            "response_json": json.dumps(RESPONSE_JSON)
        })

        # Step 2: Run the evaluator using the output of Step 1
        review_output = review_chain.invoke({
            "subject": subject,
            "quiz": quiz_output
        })

        return {
            "quiz": quiz_output,
            "review": review_output
        }
   

In [92]:
file_path=r"E:\MCQgen\data.txt"

In [93]:
with open(file_path,'r') as file:
    TEXT=file.read()

In [96]:
result = generate_mcqs(
        text=TEXT,
        number=3,
        subject="computer science",
        tone="simple"
    )
    
if result:
    print("\n" + "="*50)
    print("GENERATED QUIZ:")
    print(result["quiz"])
    print("\n" + "="*50)
    print("REVIEW:")
    print(result["review"])

ChatGoogleGenerativeAIError: Error calling model 'gemini-1.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}